**Extracción de datos a través de CSV**

In [ ]:
import pandas as pd
import json
import ast

def distill_analysis_gdl(reason_str, player, move_gdl):
    """Genera el análisis"""
    if pd.isna(reason_str) or reason_str == "":
        return f"Analysis: Strategic move based on the current GDL state at {move_gdl}."

    reason_low = str(reason_str).lower()
    player = str(player).upper()
    opponent = "O" if player == "X" else "X"

    # Extraemos la parte (R, C) para el texto
    coords = f"({move_gdl[1]}, {move_gdl[2]})"

    if "block" in reason_low:
        rule = f"The opponent ({opponent}) has a winning threat. I must block at cell {coords} to prevent their victory."
    elif "win" in reason_low or "winning" in reason_low:
        rule = f"I have two pieces aligned. I move to cell {coords} to complete the line and win."
    elif "center" in reason_low:
        rule = f"The center cell (2, 2) is the most strategic position available. I take it to control the board."
    elif "corner" in reason_low:
        rule = f"Corners like {coords} allow for creating multiple threats. Moving to this cell."
    else:
        rule = f"Analyzing GDL patterns to optimize line control. Cell {coords} is the best move to maintain pressure."

    return f"Analysis: {rule}"

def generate_expert_dataset_gdl(input_excel, output_jsonl):
    print(f"Generando D1 Expert en formato GDL puro")
    df = pd.read_excel(input_excel, sheet_name='Consolidated_Move_Records')
    df_filtered = df.dropna(subset=['board']).copy()
    df_filtered = df_filtered[df_filtered['valid'] == 1]

    dataset = []
    system_content = "You are an expert Tic-Tac-Toe strategist. Analyze the GDL board state and play optimally to win, following strategic rules and maximizing your victory probability."

    for _, row in df_filtered.iterrows():
        try:
            # Cargar datos originales del Excel (Post-jugada)
            post_board_gdl = ast.literal_eval(str(row['board']))
            move_gdl = ast.literal_eval(str(row['move'])) # Ej: ['mark', '1', '1']
            player = str(row['player']).lower() # 'x' o 'o'

            # --- LÓGICA DE TRANSFORMACIÓN A PRE-JUGADA ---
            pre_board_gdl = []
            target_r = move_gdl[1]
            target_c = move_gdl[2]

            for item in post_board_gdl:
                # Si es una celda, verificamos si es la que se acaba de marcar
                if item[0] == 'cell':
                    r, c, val = item[1], item[2], item[3]
                    if r == target_r and c == target_c:
                        # Revertimos a 'b' (blank)
                        pre_board_gdl.append(['cell', r, c, 'b'])
                    else:
                        pre_board_gdl.append(item)

                # Si es el control, lo devolvemos al jugador actual (el que hace la jugada)
                elif item[0] == 'control':
                    pre_board_gdl.append(['control', player])

                else:
                    pre_board_gdl.append(item)

            # --------------------------------------------

            legal_moves_gdl = ast.literal_eval(str(row['legalMoves']))

            try:
                reason_raw = row['reason']
                reason_dict = json.loads(reason_raw) if not pd.isna(reason_raw) else {}
                raw_explanation = reason_dict.get("explanation", reason_dict.get("reason", ""))
            except:
                raw_explanation = ""

            # Input con el board PRE-JUGADA
            user_content = f"GDL Board State: {pre_board_gdl}\nLegal Moves: {legal_moves_gdl}\nAction for Player: {player.upper()}"

            analysis = distill_analysis_gdl(raw_explanation, player, move_gdl)
            assistant_content = f"{analysis}\nMove: {move_gdl}"

            # Formato ALPACA
            dataset.append({
                "instruction": system_content,
                "input": user_content,
                "output": assistant_content
            })
        except Exception as e:
            continue

    with open(output_jsonl, 'w', encoding='utf-8') as f:
        for entry in dataset:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')

    print(f"¡Éxito! D1 GDL generado con {len(dataset)} ejemplos corregidos (Pre-jugada).")

if __name__ == "__main__":
    # Ajusta la ruta a tu carpeta de Drive
    ruta_input = '/content/drive/MyDrive/TESIS/Qwen_TicTacToe_Suicide/Consolidated_Move_Records_en_tabla_TicTacToe.xlsx'
    generate_expert_dataset_gdl(ruta_input, 'D1_Expert_TicTacToe.jsonl')

Generando D1 Expert en formato GDL puro
¡Éxito! D1 GDL generado con 1033 ejemplos corregidos (Pre-jugada).


**Extracción de datos a través de MiniMax**

In [ ]:
5#╔══════════════════════════════════════════════════════════════════╗
# ║   Tic-Tac-Toe TVN — Generador + Muestreador                      ║
# ║   Genera N estados únicos y extrae N muestras estratificadas║
# ╚══════════════════════════════════════════════════════════════════╝

# ───────────────────────────────────────────
# Generador del dataset completo Minimax
# ───────────────────────────────────────────
import json, random
from collections import defaultdict

FULL_FILE = "Expert_Dataset_TicTacToe_FULL.jsonl"
X_GDL, O_GDL, EMPTY_GDL = "x", "o", "b"

def check_winner(board):
    win_configs = [[0,1,2],[3,4,5],[6,7,8],
                   [0,3,6],[1,4,7],[2,5,8],
                   [0,4,8],[2,4,6]]
    for cfg in win_configs:
        if board[cfg[0]] == board[cfg[1]] == board[cfg[2]] != EMPTY_GDL:
            return board[cfg[0]]
    return "TIE" if EMPTY_GDL not in board else None

def minimax(board, depth, is_maximizing, turn, maximizing_player):
    winner = check_winner(board)
    if winner == maximizing_player:
        return 10 - depth
    if winner == (O_GDL if maximizing_player == X_GDL else X_GDL):
        return depth - 10
    if winner == "TIE":
        return 0
    if is_maximizing:
        best = -float('inf')
        for i in range(9):
            if board[i] == EMPTY_GDL:
                board[i] = turn
                best = max(best, minimax(board, depth+1, False,
                           O_GDL if turn==X_GDL else X_GDL, maximizing_player))
                board[i] = EMPTY_GDL
        return best
    else:
        best = float('inf')
        for i in range(9):
            if board[i] == EMPTY_GDL:
                board[i] = turn
                best = min(best, minimax(board, depth+1, True,
                           O_GDL if turn==X_GDL else X_GDL, maximizing_player))
                board[i] = EMPTY_GDL
        return best

def get_best_move(board, player):
    best_score, best_moves = -float('inf'), []
    for i in range(9):
        if board[i] == EMPTY_GDL:
            board[i] = player
            score = minimax(board, 0, False,
                            O_GDL if player==X_GDL else X_GDL, player)
            board[i] = EMPTY_GDL
            if score > best_score:
                best_score, best_moves = score, [i]
            elif score == best_score:
                best_moves.append(i)
    def rank(idx): return 3 if idx==4 else (2 if idx in [0,2,6,8] else 1)
    return sorted(best_moves, key=rank, reverse=True)[0]

def get_analysis_logic_gdl(board, player, move_idx):
    opponent = O_GDL if player == X_GDL else X_GDL
    r, c = (move_idx // 3) + 1, (move_idx % 3) + 1
    coords = f"({r}, {c})"

    board[move_idx] = player
    if check_winner(board) == player:
        board[move_idx] = EMPTY_GDL
        return (f"Analysis: Winning opportunity detected. Marking cell {coords} completes "
                f"a line of three, achieving a terminal state and a goal(100) victory under GDL rules.")
    board[move_idx] = EMPTY_GDL

    board[move_idx] = opponent
    if check_winner(board) == opponent:
        board[move_idx] = EMPTY_GDL
        return (f"Analysis: Defensive necessity. The opponent ({opponent.upper()}) threatens "
                f"to complete a line. Selecting cell {coords} is mandatory to block their "
                f"alignment and prevent a loss.")
    board[move_idx] = EMPTY_GDL

    if move_idx == 4:
        return (f"Analysis: Positional control. The center cell {coords} offers maximum "
                f"connectivity for horizontal, vertical, and diagonal lines, establishing "
                f"early GDL board dominance.")

    return (f"Analysis: Tactical development. Marking cell {coords} expands positional "
            f"pressure and restricts the opponent's future combinations across the 3x3 grid.")

def generate_all_states_gdl():
    print("Generando dataset completo (DFS sobre todos los estados)...")
    visited, dataset = set(), []
    sys_content = (
        "You are an expert Tic-Tac-Toe strategist. Analyze the GDL board state and play "
        "optimally to win, following strategic rules and maximizing your victory probability."
    )

    def dfs(board, turn):
        key = "".join(board) + turn
        if key in visited or check_winner(board) is not None:
            return
        visited.add(key)

        bm = get_best_move(board, turn)
        r, c = (bm // 3) + 1, (bm % 3) + 1

        board_gdl = [["cell", str((i//3)+1), str((i%3)+1), board[i]]
                     for i in range(9)] + [["control", turn]]
        legal_gdl = [["mark", str((i//3)+1), str((i%3)+1)]
                     for i in range(9) if board[i] == EMPTY_GDL]
        analysis  = get_analysis_logic_gdl(board, turn, bm)

        dataset.append({
            "instruction": sys_content,
            "input":       (f"GDL Board State: {board_gdl}\n"
                            f"Legal Moves: {legal_gdl}\n"
                            f"Action for Player: {turn.upper()}"),
            "output":      f"{analysis}\nMove: {['mark', str(r), str(c)]}"
        })

        next_turn = O_GDL if turn == X_GDL else X_GDL
        for i in range(9):
            if board[i] == EMPTY_GDL:
                board[i] = turn
                dfs(board, next_turn)
                board[i] = EMPTY_GDL

    dfs([EMPTY_GDL] * 9, X_GDL)
    return dataset

# Generar
full_data = generate_all_states_gdl()

# Guardar dataset completo
with open(FULL_FILE, "w", encoding="utf-8") as f:
    for entry in full_data:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"✓ Dataset completo generado: {len(full_data)} entradas → {FULL_FILE}")

✓ Configuración lista
  Total muestras : 1000
  Seed           : 42
  Salida         : Expert_Dataset_TicTacToe.jsonl
Generando dataset completo (DFS sobre todos los estados)...
✓ Dataset completo generado: 4520 entradas → Expert_Dataset_TicTacToe_FULL.jsonl
Distribución real del dataset fuente:
  Categoría              Entradas       %
  ────────────────────────────────────────
  victory                    2358   52.2%
  strategic_center            313    6.9%
  defensive                  1393   30.8%
  neutral                     456   10.1%

Plan de muestreo para 1000 ejemplos:
  Categoría               Target       %   Disponibles  Estado
  ─────────────────────────────────────────────────────────────────
  victory                    400     40%          2358  ✓
  strategic_center           300     30%           313  ✓
  defensive                  200     20%          1393  ✓
  neutral                    100     10%           456  ✓

✓ Dataset de 1000 muestras guardado → Expert_Dat

In [ ]:
import json
import random

CATEGORY_KEYWORDS = {
    "victory":   "complete the line and win",
    "defensive": "has a winning threat",
}

def classify(entry):
    output = entry.get("output", "").lower()
    for cat, kw in CATEGORY_KEYWORDS.items():
        if kw in output:
            return cat
    return None

def load_jsonl(path):
    entries = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                entries.append(json.loads(line))
    return entries

# Cargar ambos datasets
minimax = load_jsonl("dataset_final_filtrado.jsonl")
d1      = load_jsonl("D1_Expert_TicTacToe.jsonl")

# Extraer solo victory y defensive de D1
extras = [e for e in d1 if classify(e) in ("victory", "defensive")]

# Combinar y mezclar
merged = minimax + extras

# Guardar
with open("D3_Merged_TTT.jsonl", "w", encoding="utf-8") as f:
    for entry in merged:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

# Reporte
print(f"Minimax original : {len(minimax):>5} entradas")
print(f"Victory + Def D1 : {len(extras):>5} entradas")
print(f"Total final      : {len(merged):>5} entradas")

Minimax original :  3020 entradas
Victory + Def D1 :   452 entradas
Total final      :  3472 entradas


**Realizar Muestreo**

In [ ]:
import json
import random
from collections import defaultdict

FULL_FILE = "Expert_Dataset_TicTacToe_FULL.jsonl"

print(f"Cargando dataset desde: {FULL_FILE}")

full_data = []

with open(FULL_FILE, "r", encoding="utf-8") as f:
    for line in f:
        full_data.append(json.loads(line))

# ─────────────────────────────────────────────────────────────────────
# Configuración
# ─────────────────────────────────────────────────────────────────────
TOTAL_SAMPLES  = 1500    # Total de muestras a extraer
RANDOM_SEED    = 42     # Semilla para reproducibilidad
OUTPUT_FILE    = "Expert_Dataset_TicTacToe_1500.jsonl"

# Proporciones objetivo
TARGET_FRACTIONS = {
    "victory":           0.40,
    "strategic_center":  0.20,
    "defensive":         0.30,
    "neutral":           0.10,
}

print("✓ Configuración lista")
print(f"  Total muestras : {TOTAL_SAMPLES}")
print(f"  Seed           : {RANDOM_SEED}")
print(f"  Salida         : {OUTPUT_FILE}")


# ─────────────────────────────────────────────────────────────────────
# Clasificación y reporte de distribución
# ─────────────────────────────────────────────────────────────────────

CATEGORY_KEYWORDS = {
    "victory":          ["Winning opportunity", "complete the line and win"],
    "strategic_center": ["Positional control"],
    "defensive":        ["Defensive necessity", "has a winning threat"],
    "neutral":          ["Tactical development", "Analyzing GDL patterns to optimize line control"]
}

def classify(entry):
    for cat, keywords_list in CATEGORY_KEYWORDS.items():
        for keyword_str in keywords_list:
            if keyword_str in entry["output"]:
                return cat
    return "neutral"

buckets = defaultdict(list)
for entry in full_data:
    buckets[classify(entry)].append(entry)

print("Distribución real del dataset fuente:")
print(f"  {'Categoría':<22} {'Entradas':>8}  {'%':>6}")
print("  " + "─" * 40)
for cat in TARGET_FRACTIONS:
    n   = len(buckets[cat])
    pct = n / len(full_data) * 100
    print(f"  {cat:<22} {n:>8}  {pct:>5.1f}%")

# ─────────────────────────────────────────────────────────────────────
# Muestreo estratificado
# ─────────────────────────────────────────────────────────────────────

rng = random.Random(RANDOM_SEED)

# Calcular targets exactos (ajuste de redondeo en 'victory')
targets = {cat: round(TOTAL_SAMPLES * frac) for cat, frac in TARGET_FRACTIONS.items()}
diff = TOTAL_SAMPLES - sum(targets.values())
targets["victory"] += diff

print(f"\nPlan de muestreo para {TOTAL_SAMPLES} ejemplos:")
print(f"  {'Categoría':<22} {'Target':>7}  {'%':>6}  {'Disponibles':>12}  Estado")
print("  " + "─" * 65)
for cat, t in targets.items():
    avail  = len(buckets[cat])
    pct    = t / TOTAL_SAMPLES * 100
    status = "✓" if avail >= t else f"⚠  solo {avail} — se toman todos"
    print(f"  {cat:<22} {t:>7}  {pct:>5.0f}%  {avail:>12}  {status}")

# Muestrear
sampled = []
actual  = {}
for cat, t in targets.items():
    pool   = buckets[cat]
    n      = min(t, len(pool))
    chosen = rng.sample(pool, n)
    sampled.extend(chosen)
    actual[cat] = n

rng.shuffle(sampled)

# Guardar dataset
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for entry in sampled:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"\n✓ Dataset de {len(sampled)} muestras guardado → {OUTPUT_FILE}")
print("\nComposición final:")
for cat, n in actual.items():
    print(f"  {cat:<22} {n:>3}  ({n / len(sampled) * 100:.1f}%)")

# ─────────────────────────────────────────────────────────────────────
# Guardar el archivo en drive
# ─────────────────────────────────────────────────────────────────────
import os
import shutil
from google.colab import drive
# 1. Conectar Google Drive
drive.mount('/content/drive')

# 2. Definir rutas
archivo_en_colab1 = OUTPUT_FILE
archivo_en_colab2 = FULL_FILE
carpeta_en_drive = "/content/drive/MyDrive/TESIS/Qwen_TicTacToe_Suicide/TVN/"

# 3. Crear la carpeta en Drive si no existe
os.makedirs(carpeta_en_drive, exist_ok=True)

# 4. Copiar archivos
ruta_destino = os.path.join(carpeta_en_drive, os.path.basename(archivo_en_colab1))
shutil.copy(archivo_en_colab1, ruta_destino)

ruta_destino = os.path.join(carpeta_en_drive, os.path.basename(archivo_en_colab2))
shutil.copy(archivo_en_colab2, ruta_destino)

print(f"¡Archivo copiado exitosamente a: {ruta_destino}!")

Cargando dataset desde: D3_Merged_TTT.jsonl
✓ Configuración lista
  Total muestras : 100
  Seed           : 42
  Salida         : Expert_Dataset_TicTacToe_Test_100.jsonl
Distribución real del dataset fuente:
  Categoría              Entradas       %
  ────────────────────────────────────────
  victory                    1976   56.9%
  strategic_center             13    0.4%
  defensive                  1177   33.9%
  neutral                     306    8.8%

Plan de muestreo para 100 ejemplos:
  Categoría               Target       %   Disponibles  Estado
  ─────────────────────────────────────────────────────────────────
  victory                     40     40%          1976  ✓
  strategic_center            20     20%            13  ⚠  solo 13 — se toman todos
  defensive                   30     30%          1177  ✓
  neutral                     10     10%           306  ✓

✓ Dataset de 93 muestras guardado → Expert_Dataset_TicTacToe_Test_100.jsonl

Composición final:
  victory       